# 000 - Project Overview & Roadmap

This is my running index for the whole project: what each phase is, why I'm doing it, and where to find the corresponding work. I'll keep the status table below updated as phases get finished.

Project: Diagnoses-based retrieval of radiological images using CLIP neural networks (Dohvat radioloških slika temeljem dijagnoza korištenjem CLIP neuronskih mreža) - my undergrad thesis.

What I'm building: a model that, given a diagnosis text, retrieves the radiological image(s) it corresponds to, using CLIP-style contrastive learning. I'm using the RadiologyNET dataset, collected during standard clinical practice at KBC Rijeka - image/diagnosis pairs across five modalities (CR, CT, MR, RF, XA).

Required components (from my assignment brief): a literature review of CLIP and its medical variants (MedCLIP, BiomedCLIP), dataset preparation, an implemented retrieval system, a designed evaluation procedure, interpreted results, and generic recommendations for future use.

## Why I'm fine-tuning a pretrained checkpoint instead of training from scratch

My dataset is only ~10,000 exams - too small to train a CLIP model from scratch. I'm fine-tuning a pretrained checkpoint instead, most likely BiomedCLIP, since it's already trained on biomedical image-text pairs. I'm translating the Croatian diagnoses to English myself (via Google Translate) so I can use BiomedCLIP's English-only text tower and compare against it directly as a baseline later.

## A note on notebook numbering

001-how-to.ipynb was the starter notebook I was given, showing how to load the images and diagnoses - I didn't write it as part of my own phase work, so it doesn't map onto the phase numbers below. My own phase notebooks pick up from 002 onward, one per phase, each named <number>-<phase-name>.ipynb.

## Phase list

1. Project and environment setup - done. No dedicated notebook, this was folder structure, venv, GPU-enabled PyTorch, dependencies, private GitHub repo and .gitignore setup.
2. Data exploration and filtering - done. notebooks/002-data-exploration-and-filtering.ipynb. Validated dataset integrity, checked for uninformative or boilerplate diagnoses (found none worth filtering), created an ExamID based train/val/test split.
3. Diagnosis translation and tokenization check - done. notebooks/003-translation-prep.ipynb. Translated exam-level diagnoses HR to EN via Google Sheets, measured real token length distribution with the BiomedCLIP tokenizer, decided on head+tail truncation (64+190 tokens) after comparing it against conclusion-prioritized extraction and checking prior literature.
4. Dataset and DataLoader - done. notebooks/004-dataset-and-dataloader.ipynb. Broke down the multi-png-per-id problem by modality (only really affects XA, under 5 percent of the dataset), decided on random-slice-per-epoch, pulled BiomedCLIP's real preprocessing transforms, built the Dataset/DataLoader in src/data/, verified split sizes and a real batch's image/text pairing. Note: phase 9 found the random-slice choice was also being applied to val and test, where it was never intended, and fixed it.
5. Model setup - done. notebooks/005-model-setup.ipynb. Measured real GPU memory for full versus frozen versus partial fine-tuning on the RTX 3060's 6GB, ruled out full fine-tuning (needs 9.87GB), caught a Windows shared-memory spillover case that looked like a pass but was not, decided on partial fine-tuning (last 2 blocks of each encoder), backed by ConVIRT's own precedent. Implementation in src/models/.
6. Training - done. notebooks/006-training.ipynb. Built the training loop (src/training/). A trial run caught two real bugs before any real training happened: batch 128 was not actually safe for sustained use despite passing phase 5's single-step benchmark (dropped to 96), and the loss was being computed over unnormalized embeddings, silently meaningless. Both fixed. Ran a real training run: 9 epochs, early stopping kicked in, val_loss best at epoch 3 (2.2726), checkpoint saved automatically. Note: phase 9 showed the overfitting reading in this notebook was based on validation loss, which does not track retrieval quality, so the conclusion that the model degrades past epoch 3 does not hold. A correction cell is in the notebook.
7. Evaluation - done. notebooks/007-evaluation.ipynb. Evaluated checkpoints/best.pt on the held out test split. Recall@K two orders of magnitude above random chance both directions. K-means clustering recovers modality. Manual inspection across 20 example rows: modality never wrong, misses stay coherent. Implementation in src/evaluation/. Note: the run-to-run variation this notebook blamed on bf16 was actually the random-slice bug, corrected in phase 9, and the clustering ARI figures here are not reproducible as written.
8. Baseline comparison - done. notebooks/008-baseline-comparison.ipynb. Same evaluation against zero-shot BiomedCLIP. Zero-shot is already well above chance, fine-tuning adds a large retrieval gain on top. Note: this notebook's clustering conclusion was reversed by phase 9, on image embeddings zero-shot actually clusters modality slightly better than the fine-tuned model. Retrieval conclusions stand.
9. Model tuning - done. notebooks/009-model-tuning.ipynb. Tried eight ways to beat the phase 6 checkpoint: six training variants (control, layer-wise learning rate, stronger regularisation, larger batch, false negative masking, hard negative batching) and two post-hoc techniques needing no retraining (model soup, CSLS hubness correction). Not one cleared the noise bar on validation, and every variant except the regularised one landed inside a 0.508 to 0.515 band. Best model is v2_head_lr, test Recall@10 up from 0.4414 to 0.4525 text to image and 0.5611 to 0.5726 image to text, most of which traces to changing checkpoint selection rather than to any hyperparameter. The real output was corrections and negative results: evaluation was using random image slices, which broke reproducibility and invalidated phase 8's clustering claim; validation loss diverges from retrieval quality, so phase 6's loss-based selection was costing about 1.7 points of Recall@10; weight decay was hitting LayerNorms and logit_scale; the dataloader had been starving the GPU all through phase 6; and ClipLoss was treating same-exam images as negatives of each other in about 52 percent of batches. Two results worth keeping on their own: the model soup won on validation and lost on test, which is direct proof the error bar analysis was necessary, and hard negative batching visibly changed the learning trajectory (98.8 percent of top-10 retrievals already share the query's modality, yet about 78 percent of the contrastive signal was being spent on that) without moving the endpoint. Conclusion: with translation quality ruled out and eight independent attempts converging on the same number, the ceiling here is dataset size, roughly 10,000 exams, not the method.
10. Visualization and write-up - planned. notebooks/010-...ipynb. Final plots and tables for the thesis, documentation of every phase. Note that phase 9 changed several numbers that earlier notebooks report, so the write-up should pull its figures from results/phase9_comparison.json (zero-shot, phase 6/7 checkpoint and the phase 9 winner, all measured through one identical deterministic pipeline) rather than from the phase 7 and phase 8 notebooks directly.

## How I'm keeping track of things

For every phase I keep one detailed notebook under notebooks/, matching the number above, explaining what I did and why alongside the code - not just the code itself. Whenever I finish a phase, I'll come back here and flip its status from planned to done, and fill in the real notebook filename.